# MVP Pipeline: Adaptive Augmentations for Small Objects

This notebook is the main entrypoint for testing MVP in Google Colab.

Pipeline steps:
1. Install dependencies.
2. Validate dataset structure.
3. Analyze dataset and save stats.
4. Generate adaptive augmentation policy.
5. (Optional) Run training modes.
6. (Optional) Convert to COCO and run COCOeval.

In [1]:
# Install dependencies in Colab runtime.
%pip install -q ultralytics albumentations opencv-python pycocotools pyyaml numpy pandas matplotlib tqdm gdown


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.2 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import sys

# 1) Попробуем найти проект в типовых местах
candidates = [
    Path("/content/small_objects_auto_aug"),
    Path("/content/drive/MyDrive/small_objects_auto_aug"),
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = None
for p in candidates:
    if (p / "src").exists() and (p / "configs").exists():
        PROJECT_ROOT = p
        break

# 2) Если не нашли — клонируем в /content (подставь свой URL)
if PROJECT_ROOT is None:
    import subprocess
    repo_url = "https://github.com/s44w/small_objects_auto_aug.git"  # <-- замени
    subprocess.run(["git", "clone", repo_url, "/content/small_objects_auto_aug"], check=True)
    PROJECT_ROOT = Path("/content/small_objects_auto_aug")

sys.path.insert(0, str(PROJECT_ROOT))
print("Using project root:", PROJECT_ROOT)


Using project root: /content/small_objects_auto_aug


In [3]:
# Helpers for robust dataset path checks.
from pathlib import Path

def has_yolo_structure(root: Path) -> bool:
    req = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    return all(p.exists() for p in req)


In [4]:
# Imports and configuration.
from src.utils.io import load_yaml
from src.data.visdrone_manager import validate_visdrone_yolo_structure, save_validation_report
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts

project_cfg = load_yaml(PROJECT_ROOT / 'configs' / 'project_config.yaml')

# Keep config root as fallback only; actual dataset_root is resolved in bootstrap cell.
dataset_root_cfg = Path(project_cfg['dataset']['root'])
if not dataset_root_cfg.is_absolute():
    dataset_root_cfg = PROJECT_ROOT / dataset_root_cfg

splits = tuple(project_cfg['dataset'].get('splits', ['train', 'val']))
image_extensions = tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp']))

stats_dir = PROJECT_ROOT / 'artifacts' / 'stats'
policy_dir = PROJECT_ROOT / 'artifacts' / 'policy'
stats_dir.mkdir(parents=True, exist_ok=True)
policy_dir.mkdir(parents=True, exist_ok=True)

print('Config dataset root (fallback):', dataset_root_cfg)


Config dataset root (fallback): /content/small_objects_auto_aug/datasets/visdrone_yolo


In [5]:
# HOTFIX: monkey-patch prepare_visdrone_auto in current Colab runtime
from pathlib import Path
import shutil

import src.data.visdrone_manager as vm

def _resolve_split(root: Path, split: str):
    names = {
        "train": ["VisDrone2019-DET-train", "train"],
        "val": ["VisDrone2019-DET-val", "val"],
        "test": ["VisDrone2019-DET-test-dev", "VisDrone2019-DET-test-challenge", "test"],
    }[split]
    for n in names:
        p = root / n
        if p.exists() and p.is_dir():
            return p
    for n in names:
        for p in root.rglob(n):
            if p.is_dir():
                return p
    return None

def _fallback_prepare_visdrone_auto(raw_visdrone_root, output_root):
    from PIL import Image
    raw_root = Path(raw_visdrone_root)
    out = Path(output_root)
    out.mkdir(parents=True, exist_ok=True)

    for split in ("train", "val", "test"):
        src = _resolve_split(raw_root, split)
        if src is None:
            continue

        src_img = src / "images"
        src_ann = src / "annotations"
        dst_img = out / "images" / split
        dst_lbl = out / "labels" / split
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)

        for ip in src_img.glob("*.jpg"):
            shutil.copy2(ip, dst_img / ip.name)
        for ip in src_img.glob("*.png"):
            shutil.copy2(ip, dst_img / ip.name)

        if src_ann.exists():
            for ap in src_ann.glob("*.txt"):
                img = dst_img / f"{ap.stem}.jpg"
                if not img.exists():
                    img = dst_img / f"{ap.stem}.png"
                if not img.exists():
                    continue

                with Image.open(img) as im:
                    w, h = im.width, im.height

                lines = []
                for raw in ap.read_text(encoding="utf-8").splitlines():
                    if not raw.strip():
                        continue
                    row = [x.strip() for x in raw.split(",")]
                    if len(row) < 6:
                        continue
                    if row[4] == "0":  # ignore regions
                        continue

                    x, y, bw, bh = map(float, row[:4])
                    cls = int(row[5]) - 1
                    if cls < 0:
                        continue

                    xc = (x + bw / 2) / max(1, w)
                    yc = (y + bh / 2) / max(1, h)
                    wn = bw / max(1, w)
                    hn = bh / max(1, h)
                    lines.append(f"{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

                (dst_lbl / ap.name).write_text("".join(lines), encoding="utf-8")

    # local data yaml
    names = {
        0: "pedestrian", 1: "people", 2: "bicycle", 3: "car", 4: "van",
        5: "truck", 6: "tricycle", 7: "awning-tricycle", 8: "bus", 9: "motor"
    }
    try:
        import yaml
        data_yaml = {
            "path": out.as_posix(),
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "names": names,
        }
        (out / "visdrone.yaml").write_text(
            yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True),
            encoding="utf-8",
        )
    except Exception:
        pass

    print(f"[HOTFIX] Fallback conversion complete: {out}")

vm.prepare_visdrone_auto = _fallback_prepare_visdrone_auto
prepare_visdrone_auto = vm.prepare_visdrone_auto  # overwrite local name too

print("HOTFIX applied: prepare_visdrone_auto patched")


HOTFIX applied: prepare_visdrone_auto patched


In [6]:
# Dataset bootstrap from Google Drive with cache, auto-discovery and auto-repair from ZIPs.
from pathlib import Path
import shutil
import zipfile
import tempfile
import json
from datetime import datetime, timezone

from google.colab import drive
from src.data.visdrone_manager import prepare_visdrone_auto, validate_visdrone_yolo_structure

PROJECT_ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path('/content/small_objects_auto_aug')
DEFAULT_ROOT = PROJECT_ROOT / 'datasets' / 'visdrone_yolo'

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive')
dataset_root_cfg = globals().get('dataset_root_cfg', None)

CACHE_VERSION = 'visdrone_yolo_cache_v1'
FORCE_REBUILD = False
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def count_images(images_dir: Path) -> int:
    if not images_dir.exists():
        return 0
    return sum(1 for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS)

def is_yolo_ready(root: Path, require_non_empty: bool = True) -> bool:
    req = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    if not all(p.exists() for p in req):
        return False

    if not require_non_empty:
        return True

    train_n = count_images(root / 'images' / 'train')
    val_n = count_images(root / 'images' / 'val')
    return train_n > 0 and val_n > 0

def is_raw_ready(root: Path) -> bool:
    if (root / 'VisDrone2019-DET-train').exists() and (root / 'VisDrone2019-DET-val').exists():
        return True

    train_like = list(root.glob('*DET*train*')) + list(root.glob('*train*'))
    val_like = list(root.glob('*DET*val*')) + list(root.glob('*val*'))
    if any(p.is_dir() for p in train_like) and any(p.is_dir() for p in val_like):
        return True

    return False

def find_raw_root(drive_root: Path):
    candidates = [
        drive_root / 'datasets' / 'visdrone_raw',
        drive_root / 'visdrone_raw',
        drive_root / 'datasets' / 'VisDrone',
        drive_root / 'VisDrone',
    ]
    for c in candidates:
        if c.exists() and is_raw_ready(c):
            return c

    for t in drive_root.rglob('*DET*train*'):
        if not t.is_dir():
            continue
        parent = t.parent
        if is_raw_ready(parent):
            return parent
    return None

def find_yolo_root(drive_root: Path):
    candidates = [
        drive_root / 'datasets' / 'visdrone_yolo',
        drive_root / 'visdrone_yolo',
        DEFAULT_ROOT,
        dataset_root_cfg if dataset_root_cfg is not None else None,
    ]
    for c in candidates:
        if c is not None and c.exists() and is_yolo_ready(c, require_non_empty=True):
            return c

    for images_train in drive_root.rglob('images/train'):
        root = images_train.parent.parent
        if is_yolo_ready(root, require_non_empty=True):
            return root
    return None

def rebuild_raw_from_zips(zip_root: Path, raw_root: Path) -> bool:
    if not zip_root.exists():
        return False

    zip_files = sorted(zip_root.glob('*.zip'))
    if not zip_files:
        return False

    raw_root.mkdir(parents=True, exist_ok=True)
    print(f'[INFO] Rebuilding raw structure from ZIPs: {zip_root} -> {raw_root}')

    for z in zip_files:
        split_name = z.stem
        target_dir = raw_root / split_name
        if target_dir.exists():
            print(f'[SKIP] already prepared: {target_dir.name}')
            continue

        print(f'[UNZIP] {z.name} -> {target_dir.name}')
        with tempfile.TemporaryDirectory() as td:
            tmp = Path(td)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(tmp)

            entries = [p for p in tmp.iterdir()]
            if len(entries) == 1 and entries[0].is_dir():
                shutil.move(str(entries[0]), str(target_dir))
            else:
                target_dir.mkdir(parents=True, exist_ok=True)
                for item in entries:
                    shutil.move(str(item), str(target_dir / item.name))

    return True

def load_cache_meta(meta_path: Path):
    if not meta_path.exists():
        return None
    try:
        return json.loads(meta_path.read_text(encoding='utf-8'))
    except Exception:
        return None

def save_cache_meta(meta_path: Path, root: Path):
    payload = {
        'cache_version': CACHE_VERSION,
        'updated_utc': datetime.now(timezone.utc).isoformat(),
        'dataset_root': str(root),
        'train_images': count_images(root / 'images' / 'train'),
        'val_images': count_images(root / 'images' / 'val'),
    }
    meta_path.parent.mkdir(parents=True, exist_ok=True)
    meta_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return payload

cached_root = find_yolo_root(DRIVE_ROOT)
cache_meta_path = None
cache_meta = None

if cached_root is not None:
    cache_meta_path = cached_root / '_cache_meta.json'
    cache_meta = load_cache_meta(cache_meta_path)

use_cache = (
    (not FORCE_REBUILD)
    and (cached_root is not None)
    and is_yolo_ready(cached_root, require_non_empty=True)
    and (cache_meta is not None)
    and (cache_meta.get('cache_version') == CACHE_VERSION)
)

if use_cache:
    dataset_root = cached_root
    print(f'[OK] Using cached YOLO dataset: {dataset_root}')
    print(f"[OK] Cache meta: {cache_meta_path}")
else:
    if cached_root is not None and is_yolo_ready(cached_root, require_non_empty=True):
        dataset_root = cached_root
        if cache_meta_path is None:
            cache_meta_path = dataset_root / '_cache_meta.json'
        payload = save_cache_meta(cache_meta_path, dataset_root)
        print(f'[OK] Using existing YOLO dataset and refreshing cache metadata: {dataset_root}')
        print(f"[OK] train={payload['train_images']} val={payload['val_images']}")
    else:
        raw_root = find_raw_root(DRIVE_ROOT)

        if raw_root is None:
            zip_candidates = [
                DRIVE_ROOT / 'datasets' / 'visdrone_zips',
                DRIVE_ROOT / 'visdrone_zips',
            ]
            repaired = False
            for zc in zip_candidates:
                repaired = rebuild_raw_from_zips(zc, DRIVE_ROOT / 'datasets' / 'visdrone_raw')
                if repaired:
                    break

            raw_root = find_raw_root(DRIVE_ROOT)

        if raw_root is None:
            preview = []
            ds = DRIVE_ROOT / 'datasets'
            if ds.exists():
                preview = sorted([p.name for p in ds.iterdir()])[:50]
            raise FileNotFoundError(
                'VisDrone raw/YOLO dataset not found in Drive.\n'
                f'Checked under: {DRIVE_ROOT}\n'
                f'datasets/ preview: {preview}\n'
                'Expected raw markers like *DET*train* and *DET*val*'
            )

        target_yolo = DRIVE_ROOT / 'datasets' / 'visdrone_yolo'
        target_yolo.mkdir(parents=True, exist_ok=True)

        # Clean only train/val targets before rebuilding to avoid stale or half-built cache.
        for split in ('train', 'val'):
            shutil.rmtree(target_yolo / 'images' / split, ignore_errors=True)
            shutil.rmtree(target_yolo / 'labels' / split, ignore_errors=True)

        print(f'[INFO] Converting raw VisDrone -> YOLO: {raw_root} -> {target_yolo}')
        prepare_visdrone_auto(raw_visdrone_root=raw_root, output_root=target_yolo)

        if not is_yolo_ready(target_yolo, require_non_empty=True):
            train_n = count_images(target_yolo / 'images' / 'train')
            val_n = count_images(target_yolo / 'images' / 'val')
            raise RuntimeError(
                f'YOLO conversion finished but dataset is incomplete: train={train_n}, val={val_n}. '
                'Check raw split structure in Drive.'
            )

        dataset_root = target_yolo
        cache_meta_path = dataset_root / '_cache_meta.json'
        payload = save_cache_meta(cache_meta_path, dataset_root)
        print(f"[OK] Cache saved: {cache_meta_path}")
        print(f"[OK] train={payload['train_images']} val={payload['val_images']}")

DEFAULT_ROOT.parent.mkdir(parents=True, exist_ok=True)
if dataset_root != DEFAULT_ROOT:
    if DEFAULT_ROOT.exists() or DEFAULT_ROOT.is_symlink():
        try:
            if DEFAULT_ROOT.is_symlink() or DEFAULT_ROOT.is_file():
                DEFAULT_ROOT.unlink()
            else:
                shutil.rmtree(DEFAULT_ROOT, ignore_errors=True)
        except Exception as e:
            print(f'[WARN] Could not clear default root: {e}')

    try:
        DEFAULT_ROOT.symlink_to(dataset_root, target_is_directory=True)
        print(f'[OK] Linked {DEFAULT_ROOT} -> {dataset_root}')
    except Exception as e:
        print(f'[WARN] Symlink not created: {e}')

USE_LOCAL_RUNTIME_COPY = True
LOCAL_RUNTIME_ROOT = Path('/content/datasets/visdrone_yolo')

if USE_LOCAL_RUNTIME_COPY:
    if not is_yolo_ready(LOCAL_RUNTIME_ROOT, require_non_empty=True):
        print(f'[INFO] Copying dataset to local runtime SSD: {dataset_root} -> {LOCAL_RUNTIME_ROOT}')
        if LOCAL_RUNTIME_ROOT.exists() or LOCAL_RUNTIME_ROOT.is_symlink():
            try:
                if LOCAL_RUNTIME_ROOT.is_symlink() or LOCAL_RUNTIME_ROOT.is_file():
                    LOCAL_RUNTIME_ROOT.unlink()
                else:
                    shutil.rmtree(LOCAL_RUNTIME_ROOT, ignore_errors=True)
            except Exception as e:
                print(f'[WARN] Could not clear local runtime root: {e}')

        LOCAL_RUNTIME_ROOT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(dataset_root, LOCAL_RUNTIME_ROOT)
    else:
        print(f'[OK] Reusing local runtime dataset: {LOCAL_RUNTIME_ROOT}')

    dataset_root = LOCAL_RUNTIME_ROOT
    print(f'[OK] Using local runtime dataset root: {dataset_root}')

import time

def quick_split_counts(root: Path, split: str) -> tuple[int, int]:
    images_dir = root / 'images' / split
    labels_dir = root / 'labels' / split
    img_n = count_images(images_dir)
    lbl_n = len(list(labels_dir.glob('*.txt'))) if labels_dir.exists() else 0
    return img_n, lbl_n

def safe_validate(root: Path, retries: int = 3, delay_sec: float = 2.0):
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            return validate_visdrone_yolo_structure(root, splits=('train', 'val'))
        except OSError as exc:
            last_exc = exc
            print(f'[WARN] Deep validation failed ({attempt}/{retries}): {exc}')
            if attempt < retries:
                time.sleep(delay_sec * attempt)

    train_img_n, train_lbl_n = quick_split_counts(root, 'train')
    val_img_n, val_lbl_n = quick_split_counts(root, 'val')
    print('[WARN] Falling back to quick split checks due to Drive I/O instability.')
    return {
        'dataset_root': root.as_posix(),
        'is_valid': (train_img_n > 0 and val_img_n > 0),
        'warning': f'validate_visdrone_yolo_structure failed: {last_exc}',
        'splits': {
            'train': {'num_images': train_img_n, 'num_label_files': train_lbl_n},
            'val': {'num_images': val_img_n, 'num_label_files': val_lbl_n},
        },
    }

report = safe_validate(dataset_root)
train_n = report['splits']['train']['num_images']
val_n = report['splits']['val']['num_images']
print('is_valid:', report['is_valid'])
print('train images:', train_n)
print('val images:', val_n)
print('Using dataset_root:', dataset_root)
if 'warning' in report:
    print('[WARN]', report['warning'])

assert train_n > 0 and val_n > 0, 'train/val images must be non-empty before training'





Mounted at /content/drive
[OK] Using cached YOLO dataset: /content/drive/MyDrive/datasets/visdrone_yolo
[OK] Cache meta: /content/drive/MyDrive/datasets/visdrone_yolo/_cache_meta.json
[OK] Linked /content/small_objects_auto_aug/datasets/visdrone_yolo -> /content/drive/MyDrive/datasets/visdrone_yolo
is_valid: True
train images: 1610
val images: 548
Using dataset_root: /content/drive/MyDrive/datasets/visdrone_yolo


In [7]:
# Quick smoke check (fast): subset -> analyze -> policy -> artifact assertions.
# This is enough to verify project operability without heavy training/eval.
from pathlib import Path
import json

from src.data.subset_builder import build_yolo_subset
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.data.visdrone_manager import validate_visdrone_yolo_structure

RUN_SMOKE = True

if RUN_SMOKE:
    SMOKE_TRAIN_IMAGES = 60
    SMOKE_VAL_IMAGES = 20
    SMOKE_SPLITS = ('train',)
    SMOKE_SEED = 42

    subset_root = PROJECT_ROOT / 'datasets' / 'visdrone_smoke'
    subset_report = build_yolo_subset(
        dataset_root=dataset_root,
        output_root=subset_root,
        train_images=SMOKE_TRAIN_IMAGES,
        val_images=SMOKE_VAL_IMAGES,
        seed=SMOKE_SEED,
        clean_output=True,
    )
    print('subset:', subset_report)

    val_report = validate_visdrone_yolo_structure(subset_root, splits=('train', 'val'))
    print('subset valid:', val_report['is_valid'])

    smoke_stats_dir = PROJECT_ROOT / 'artifacts' / 'smoke' / 'stats'
    analyzer_cfg = DatasetAnalyzerConfig(
        small_max_area=float(project_cfg['analysis']['small_max_area']),
        medium_max_area=float(project_cfg['analysis']['medium_max_area']),
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
        image_extensions=tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
        generate_plots=False,
    )
    smoke_stats = analyze_dataset(
        dataset_root=subset_root,
        output_dir=smoke_stats_dir,
        splits=SMOKE_SPLITS,
        config=analyzer_cfg,
    )

    smoke_policy_dir = PROJECT_ROOT / 'artifacts' / 'smoke' / 'policy'
    rule_cfg = RuleEngineConfig.from_project_config(project_cfg)
    smoke_policy, smoke_decision_report = generate_policy_from_stats(smoke_stats, cfg=rule_cfg)
    smoke_paths = save_policy_artifacts(smoke_policy, smoke_decision_report, output_dir=smoke_policy_dir)

    required_files = [
        smoke_stats_dir / 'dataset_stats.json',
        smoke_stats_dir / 'dataset_stats.csv',
        smoke_policy_dir / 'policy_adaptive.json',
        smoke_policy_dir / 'policy_adaptive.yaml',
        smoke_policy_dir / 'decision_report.json',
    ]
    for p in required_files:
        assert p.exists(), f'Missing artifact: {p}'

    print('\n[OK] Smoke check passed')
    print('dataset_stats:', smoke_stats_dir / 'dataset_stats.json')
    print('policy_json:', smoke_policy_dir / 'policy_adaptive.json')
    print('decision_report:', smoke_policy_dir / 'decision_report.json')
    print('selected albu transforms:', [x['name'] for x in smoke_policy['albumentations_spec']])
    print('ultralytics args:', json.dumps(smoke_policy['ultralytics_args'], ensure_ascii=False, indent=2))
else:
    print('Smoke is skipped. Set RUN_SMOKE=True to enable.')


subset: SubsetBuildReport(train_images=60, val_images=20, output_root='/content/small_objects_auto_aug/datasets/visdrone_smoke')
subset valid: True


Analyze train:   0%|          | 0/60 [00:00<?, ?img/s]


[OK] Smoke check passed
dataset_stats: /content/small_objects_auto_aug/artifacts/smoke/stats/dataset_stats.json
policy_json: /content/small_objects_auto_aug/artifacts/smoke/policy/policy_adaptive.json
decision_report: /content/small_objects_auto_aug/artifacts/smoke/policy/decision_report.json
selected albu transforms: ['BBoxAwareCrop', 'BBoxCopyPaste']
ultralytics args: {
  "close_mosaic": 10,
  "degrees": 5.0,
  "mixup": 0.0,
  "hsv_v": 0.5,
  "hsv_h": 0.015,
  "scale": 0.3,
  "fliplr": 0.5,
  "cutmix": 0.0,
  "mosaic": 0.7,
  "perspective": 0.0,
  "flipud": 0.0,
  "translate": 0.05,
  "hsv_s": 0.6
}


In [8]:
# Step 1-2 (optional full run): validate and analyze dataset.
# Default is quality-oriented: use full dataset for stronger policy/training signal.
RUN_FULL_ANALYSIS = True
USE_SUBSET_FOR_FULL_PIPELINE = False
SUBSET_TRAIN_IMAGES = 5000
SUBSET_VAL_IMAGES = 1000
SUBSET_SEED = 42
GENERATE_PLOTS_FOR_FULL_ANALYSIS = False

if RUN_FULL_ANALYSIS:
    from src.data.subset_builder import build_yolo_subset

    pipeline_dataset_root = dataset_root

    if USE_SUBSET_FOR_FULL_PIPELINE:
        subset_root = PROJECT_ROOT / 'datasets' / 'visdrone_full_subset'
        subset_report = build_yolo_subset(
            dataset_root=dataset_root,
            output_root=subset_root,
            train_images=SUBSET_TRAIN_IMAGES,
            val_images=SUBSET_VAL_IMAGES,
            seed=SUBSET_SEED,
            clean_output=True,
        )
        pipeline_dataset_root = subset_root
        print('[INFO] Full pipeline subset prepared:', subset_report)
    else:
        print('[INFO] Using full dataset for full pipeline run.')

    print('[INFO] Analysis dataset root:', pipeline_dataset_root)

    validation_report = validate_visdrone_yolo_structure(
        dataset_root=pipeline_dataset_root,
        splits=splits,
        image_extensions=image_extensions,
    )
    save_validation_report(validation_report, stats_dir / 'validation_report.json')
    print('Validation is_valid =', validation_report['is_valid'])

    analyzer_cfg = DatasetAnalyzerConfig(
        small_max_area=float(project_cfg['analysis']['small_max_area']),
        medium_max_area=float(project_cfg['analysis']['medium_max_area']),
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
        image_extensions=image_extensions,
        generate_plots=GENERATE_PLOTS_FOR_FULL_ANALYSIS,
        show_progress=True,
        progress_log_every=200,
    )
    stats = analyze_dataset(
        dataset_root=pipeline_dataset_root,
        output_dir=stats_dir,
        splits=splits,
        config=analyzer_cfg,
    )
    print('Stats saved to:', stats_dir)
else:
    print('Full analysis is skipped. Set RUN_FULL_ANALYSIS=True to enable.')



[INFO] Full pipeline subset prepared: SubsetBuildReport(train_images=1610, val_images=548, output_root='/content/small_objects_auto_aug/datasets/visdrone_full_subset')
[INFO] Analysis dataset root: /content/small_objects_auto_aug/datasets/visdrone_full_subset
Validation is_valid = True


Analyze train:   0%|          | 0/1610 [00:00<?, ?img/s]

Analyze val:   0%|          | 0/548 [00:00<?, ?img/s]

Stats saved to: /content/small_objects_auto_aug/artifacts/stats


In [9]:
# Step 3 (optional full run): generate adaptive policy from full stats.
RUN_FULL_POLICY = True

if RUN_FULL_POLICY:
    rule_cfg = RuleEngineConfig.from_project_config(project_cfg)
    policy, decision_report = generate_policy_from_stats(stats, cfg=rule_cfg)
    paths = save_policy_artifacts(policy=policy, decision_report=decision_report, output_dir=policy_dir)

    print('Policy JSON:', paths['policy_json'])
    print('Policy YAML:', paths['policy_yaml'])
    print('Decision report:', paths['decision_report'])
    print('Fired rules:', len(decision_report.get('fired_rules', [])))
else:
    print('Full policy generation is skipped. Set RUN_FULL_POLICY=True to enable.')


Policy JSON: /content/small_objects_auto_aug/artifacts/policy/policy_adaptive.json
Policy YAML: /content/small_objects_auto_aug/artifacts/policy/policy_adaptive.yaml
Decision report: /content/small_objects_auto_aug/artifacts/policy/decision_report.json
Fired rules: 6


In [10]:
# Step 3.5 (optional): ensure YOLO dataset integrity / optional rebuild from raw.
from pathlib import Path

from src.data.visdrone_manager import prepare_visdrone_auto, validate_visdrone_yolo_structure

RAW_ROOT = Path('/content/drive/MyDrive/datasets/visdrone_raw')
YOLO_ROOT = Path('/content/drive/MyDrive/datasets/visdrone_yolo')
FORCE_REBUILD_FROM_RAW = False

if FORCE_REBUILD_FROM_RAW:
    print(f'[INFO] Rebuilding YOLO dataset from raw: {RAW_ROOT} -> {YOLO_ROOT}')
    prepare_visdrone_auto(raw_visdrone_root=RAW_ROOT, output_root=YOLO_ROOT)

report = validate_visdrone_yolo_structure(YOLO_ROOT, splits=('train', 'val'))
train_n = report['splits']['train']['num_images']
val_n = report['splits']['val']['num_images']
print('YOLO root:', YOLO_ROOT)
print('is_valid:', report['is_valid'])
print('train images:', train_n)
print('val images:', val_n)

# Sanity checks: prevent silent training on broken/partial data.
assert train_n > 0 and val_n > 0, 'Train/Val images are empty.'
if train_n < 3000:
    print('[WARN] train split is unexpectedly small for full VisDrone. Check raw split mapping.')

# Keep notebook-wide dataset_root consistent.
dataset_root = YOLO_ROOT
print('[INFO] dataset_root set to:', dataset_root)



[train] source: /content/drive/MyDrive/datasets/visdrone_raw (images=1610)
[train] copied images: 1610
[val] source: /content/drive/MyDrive/datasets/visdrone_raw/VisDrone2019-DET-val (images=548)
[val] copied images: 548
train images: 1610
val images: 548
runtime yaml: /content/small_objects_auto_aug/artifacts/runtime_visdrone.yaml


In [11]:
# Step 4 (optional): run training suite
# Quality defaults below are intentionally heavier and may take around 45-90 minutes on a Colab T4.
from pathlib import Path
import yaml

RUN_TRAINING = True
TRAIN_PROFILE = 'hour'  # 'fast' | 'balanced' | 'quality' | 'hour' | 'max_quality'
MODEL_OVERRIDE = 'yolo11n.pt'  # Set to 'yolo11s.pt' for potentially better quality (slower)
RUN_ABLATION = False

if RUN_TRAINING:
    import importlib
    import src.training.train_runner as train_runner_module

    importlib.reload(train_runner_module)
    TrainRunConfig = train_runner_module.TrainRunConfig
    run_mvp_training_suite = train_runner_module.run_mvp_training_suite

    train_dataset_root = globals().get('pipeline_dataset_root', dataset_root)

    def _count_split_images(root: Path, split: str) -> int:
        exts = {'.jpg', '.jpeg', '.png', '.bmp'}
        images_dir = root / 'images' / split
        if not images_dir.exists():
            return 0
        return sum(1 for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in exts)

    train_n = _count_split_images(train_dataset_root, 'train')
    val_n = _count_split_images(train_dataset_root, 'val')
    print(f'[INFO] Training dataset root: {train_dataset_root}')
    print(f'[INFO] train images: {train_n}, val images: {val_n}')
    assert train_n > 0 and val_n > 0, 'Training dataset has empty train/val split.'

    profile_map = {
        'fast': {
            'epochs': 5,
            'imgsz': 640,
            'batch': 8,
            'workers': 2,
            'fraction': 0.2,
            'multi_scale': False,
        },
        'balanced': {
            'epochs': 15,
            'imgsz': 768,
            'batch': 8,
            'workers': 2,
            'fraction': 1.0,
            'multi_scale': True,
        },
        'quality': {
            'epochs': 25,
            'imgsz': 896,
            'batch': 8,
            'workers': 2,
            'fraction': 1.0,
            'multi_scale': True,
        },
        'hour': {
            'epochs': 40,
            'imgsz': 960,
            'batch': 6,
            'workers': 2,
            'fraction': 1.0,
            'multi_scale': True,
        },
        'max_quality': {
            'epochs': 60,
            'imgsz': 1024,
            'batch': 4,
            'workers': 2,
            'fraction': 1.0,
            'multi_scale': True,
        },
    }
    assert TRAIN_PROFILE in profile_map, f'Unknown TRAIN_PROFILE: {TRAIN_PROFILE}'
    prof = profile_map[TRAIN_PROFILE]
    print(f'[INFO] Training profile: {TRAIN_PROFILE} -> {prof}')

    runtime_data_yaml = PROJECT_ROOT / 'artifacts' / 'runtime_visdrone.yaml'
    runtime_data_yaml.parent.mkdir(parents=True, exist_ok=True)

    names = project_cfg['dataset']['visdrone_classes']
    data_cfg = {
        'path': str(train_dataset_root),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': {i: n for i, n in enumerate(names)},
    }
    runtime_data_yaml.write_text(
        yaml.safe_dump(data_cfg, sort_keys=False, allow_unicode=True),
        encoding='utf-8',
    )
    print(f'[INFO] Runtime data.yaml: {runtime_data_yaml}')

    # Create stronger runtime training args for baseline/manual references.
    runtime_baseline_yaml = PROJECT_ROOT / 'artifacts' / 'runtime_baseline.yaml'
    runtime_manual_yaml = PROJECT_ROOT / 'artifacts' / 'runtime_manual.yaml'

    baseline_args = {
        'mosaic': 0.9,
        'close_mosaic': 15,
        'mixup': 0.0,
        'cutmix': 0.0,
        'hsv_h': 0.015,
        'hsv_s': 0.7,
        'hsv_v': 0.45,
        'degrees': 2.0,
        'translate': 0.08,
        'scale': 0.55,
        'perspective': 0.0,
        'fliplr': 0.5,
        'flipud': 0.0,
    }
    manual_args = {
        'mosaic': 0.85,
        'close_mosaic': 15,
        'mixup': 0.0,
        'cutmix': 0.0,
        'hsv_h': 0.015,
        'hsv_s': 0.6,
        'hsv_v': 0.5,
        'degrees': 5.0,
        'translate': 0.08,
        'scale': 0.45,
        'perspective': 0.0,
        'fliplr': 0.5,
        'flipud': 0.0,
    }
    runtime_baseline_yaml.write_text(yaml.safe_dump(baseline_args, sort_keys=False), encoding='utf-8')
    runtime_manual_yaml.write_text(yaml.safe_dump(manual_args, sort_keys=False), encoding='utf-8')

    full_policy = policy_dir / 'policy_adaptive.json'
    smoke_policy = PROJECT_ROOT / 'artifacts' / 'smoke' / 'policy' / 'policy_adaptive.json'
    adaptive_policy_json = full_policy if full_policy.exists() else smoke_policy
    assert adaptive_policy_json.exists(), f'Policy JSON not found: {adaptive_policy_json}'

    model_name = str(MODEL_OVERRIDE or project_cfg['training']['model'])

    cfg_kwargs = dict(
        data_yaml=str(runtime_data_yaml),
        model=model_name,
        epochs=int(prof['epochs']),
        imgsz=int(prof['imgsz']),
        batch=int(prof['batch']),
        device=0,
        workers=int(prof['workers']),
        fraction=float(prof['fraction']),
        project_dir=str(PROJECT_ROOT / 'runs'),
        seed=int(project_cfg['project']['seed']),
        deterministic=bool(project_cfg['project']['deterministic']),
        rect=bool(project_cfg['training'].get('rect', False)),
        multi_scale=bool(prof['multi_scale']),
        baseline_disable_albumentations=bool(project_cfg['training'].get('baseline_disable_albumentations', True)),
    )
    if hasattr(TrainRunConfig, '__dataclass_fields__') and 'plots' in TrainRunConfig.__dataclass_fields__:
        cfg_kwargs['plots'] = False

    train_cfg = TrainRunConfig(**cfg_kwargs)

    run_dirs = run_mvp_training_suite(
        config=train_cfg,
        baseline_yaml_path=runtime_baseline_yaml,
        manual_yaml_path=runtime_manual_yaml,
        adaptive_policy_json_path=adaptive_policy_json,
        run_ablation=RUN_ABLATION,
    )
    print(run_dirs)
else:
    print('Training is skipped. Set RUN_TRAINING=True to enable.')



[INFO] Training dataset root: /content/small_objects_auto_aug/datasets/visdrone_full_subset
[INFO] train images: 1610, val images: 548
[INFO] Runtime data.yaml: /content/small_objects_auto_aug/artifacts/runtime_visdrone.yaml
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[], auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/small_objects_auto_aug/artifacts/runtime_visdrone.yaml, deg

Exception in thread Thread-33 (plot_images):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/utils/plotting.py", line 796, in plot_images
    annotator.box_label(box, label, color=color)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/utils/plotting.py", line 326, in box_label
    ) if multi_points else self.draw.rectangle(box, width=self.lw, outline=color)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/ImageDraw.py", line 398, in rectangle
    self.draw.draw_rectangle(xy, ink, 0, width)
ValueError: x1 must be greater than or equal to x0


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 3.4it/s 10.3s
                   all        548      38759      0.581     0.0592     0.0405      0.022
            pedestrian        520       8844      0.273      0.104     0.0702     0.0279
                people        482       5125          0          0    0.00113   0.000281
               bicycle        364       1287          1          0          0          0
                   car        515      14064      0.311      0.452        0.3       0.17
                   van        421       1975          1          0      0.012    0.00718
                 truck        266        750      0.105      0.032     0.0141     0.0094
              tricycle        337       1045          1          0    4.2e-05   1.68e-05
       awning-tricycle        220        532          1          0          0          0
                   bus        131        251      0.124    0.00398     

RepresenterError: ('cannot represent an object', BBoxAwareCropTransform(height=640, width=640, min_visibility=0.3, min_area=16.0, p=0.6, seed=42))

In [12]:
# Step 5 (optional): inference on val + COCO conversion + COCOeval
RUN_EVAL = True
USE_TTA = True
EVAL_IMGSZ = 960

if RUN_EVAL:
    import pandas as pd
    from ultralytics import YOLO
    from src.evaluation.coco_converter import convert_yolo_gt_to_coco, convert_yolo_pred_txt_to_coco
    from src.evaluation.coco_eval_runner import run_coco_eval
    from src.evaluation.metrics_report import build_markdown_report

    eval_dataset_root = globals().get('pipeline_dataset_root', dataset_root)
    print(f'[INFO] Eval dataset root: {eval_dataset_root}')

    eval_dir = PROJECT_ROOT / 'artifacts' / 'eval'
    eval_dir.mkdir(parents=True, exist_ok=True)

    def _pick_best_run_by_val_metric(run_names: list[str]):
        best = None
        for name in run_names:
            run_dir = PROJECT_ROOT / 'runs' / name
            weights = run_dir / 'weights' / 'best.pt'
            results_csv = run_dir / 'results.csv'
            if not weights.exists():
                continue

            score = -1.0
            if results_csv.exists():
                try:
                    df = pd.read_csv(results_csv)
                    if 'metrics/mAP50-95(B)' in df.columns:
                        score = float(df['metrics/mAP50-95(B)'].max())
                except Exception:
                    score = -1.0

            if best is None or score > best['score']:
                best = {'name': name, 'weights': weights, 'score': score}

        return best

    run_candidates = ['adaptive', 'manual', 'baseline', 'adaptive_no_mosaic', 'adaptive_no_custom_albu']
    best = _pick_best_run_by_val_metric(run_candidates)
    assert best is not None, 'No trained weights found in runs/*/weights/best.pt'

    best_weights = best['weights']
    run_used = best['name']
    print(f"[INFO] Using run '{run_used}' for eval (best val mAP50-95={best['score']:.5f})")

    pred_project = PROJECT_ROOT / 'runs' / 'eval_predict'
    model = YOLO(str(best_weights))
    model.predict(
        source=str(eval_dataset_root / 'images' / 'val'),
        device=0,
        imgsz=EVAL_IMGSZ,
        conf=0.001,
        iou=0.6,
        max_det=500,
        augment=USE_TTA,
        save=False,
        save_txt=True,
        save_conf=True,
        project=str(pred_project),
        name=run_used,
        exist_ok=True,
        verbose=False,
    )

    pred_labels_dir = pred_project / run_used / 'labels'
    assert pred_labels_dir.exists(), f'Prediction labels not found: {pred_labels_dir}'

    coco_gt_path = eval_dir / 'coco_gt.json'
    coco_dt_path = eval_dir / 'coco_dt.json'

    convert_yolo_gt_to_coco(
        images_dir=eval_dataset_root / 'images' / 'val',
        labels_dir=eval_dataset_root / 'labels' / 'val',
        class_names=project_cfg['dataset']['visdrone_classes'],
        output_path=coco_gt_path,
        image_extensions=image_extensions,
    )
    convert_yolo_pred_txt_to_coco(
        pred_labels_dir=pred_labels_dir,
        images_dir=eval_dataset_root / 'images' / 'val',
        output_path=coco_dt_path,
        image_extensions=image_extensions,
    )

    metrics = run_coco_eval(
        coco_gt_path=coco_gt_path,
        coco_dt_path=coco_dt_path,
        output_path=eval_dir / 'coco_eval.json',
        use_tiny_eval=True,
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
    )

    report_path = build_markdown_report(
        {f'{run_used}_eval': metrics},
        PROJECT_ROOT / 'artifacts' / 'reports' / 'mvp_report.md'
    )
    print('Run used:', run_used)
    print(metrics)
    print('Report:', report_path)
else:
    print('Evaluation is skipped. Set RUN_EVAL=True to enable.')



[INFO] Eval dataset root: /content/small_objects_auto_aug/datasets/visdrone_full_subset
Results saved to /content/small_objects_auto_aug/runs/eval_predict/adaptive
548 labels saved to /content/small_objects_auto_aug/runs/eval_predict/adaptive/labels
loading annotations into memory...
Done (t=0.11s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.28s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=37.68s).
Accumulating evaluation results...
DONE (t=1.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.021
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.039
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.021
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.009
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.039
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= larg